# NumPy Math & Aggregation

> 📘 **Python Mastery** · Module 10 — NumPy · Lesson 5/7

This is where NumPy starts feeling like a pocket calculator for matrices: element-wise math, universal functions, and the aggregations (sum, mean, std...) that summarize arrays — including the moment `axis` finally clicks.

## 🎯 Learning Objectives

- Apply arithmetic operators and scalars element-wise across arrays
- Use universal functions: `sqrt`, `exp`, `log`, `abs`, `sin`, `round`
- Produce boolean arrays from comparisons and count with `(mask).mean()`
- Summarize arrays with `sum`, `mean`, `std`, `var`, `min`, `max`, `median`, `percentile`
- Explain exactly what `axis=0` and `axis=1` do to a matrix
- Locate extremes with `argmin`/`argmax`, accumulate with `cumsum`/`cumprod`, test with `any`/`all`, and handle NaN with nan-aware functions

## 1. Element-Wise Arithmetic

Every operator (`+ - * / ** % //`) applies position by position. Two same-shaped arrays combine pairwise; a scalar is applied to every element (that is broadcasting from last lesson).

**Syntax:**
```python
a + b      # element-wise between two arrays
a * 2      # scalar to every element
a ** 2     # power of every element
```

In [ ]:
import numpy as np

a = np.array([10, 20, 30])
b = np.array([1, 2, 3])

print("a + b  :", a + b)       # 10+1, 20+2, 30+3
print("a - b  :", a - b)
print("a * b  :", a * b)       # NOT matrix multiplication - just pairwise
print("a / b  :", a / b)       # true division -> floats
print("a // b :", a // b)      # floor division
print("a % 7  :", a % 7)       # remainder of every element
print("a ** 2 :", a ** 2)
print("a + 100:", a + 100)     # scalar broadcast everywhere

## 2. Universal Functions (ufuncs)

Functions that act element-by-element are called **universal functions**. They are compiled C loops wearing Python clothes: fast, vectorized, and they broadcast like the operators above.

**Syntax:**
```python
np.sqrt(arr)   # square root of every element
np.exp(arr)    # e^x
np.abs(arr)    # absolute value
np.sin(arr)    # trigonometry (radians!)
np.round(arr)  # rounding
```

In [ ]:
import numpy as np

squares = np.array([0, 4, 9, 16, 25])

print("sqrt :", np.sqrt(squares))
print("exp  :", np.round(np.exp([0, 1, 2]), 3))          # e^0, e^1, e^2
print("log  :", np.round(np.log([1, np.e, np.e**2]), 3))
print("abs  :", np.abs(np.array([-5, 3, -0.5])))
print("sin  :", np.round(np.sin([0, np.pi / 2, np.pi]), 3))   # radians!

In [ ]:
import numpy as np

values = np.array([2.1, 2.5, 2.7, 3.5, -2.5])

print("round :", np.round(values))   # banker's rounding: .5 ties go to the EVEN side!
print("floor :", np.floor(values))   # always down
print("ceil  :", np.ceil(values))    # always up
print("trunc :", np.trunc(values))   # chop off the fraction

## 3. Comparisons Return Boolean Arrays

Comparing an array with a value does not give one True/False — it gives a verdict PER ELEMENT. Those boolean arrays are masks (previous lesson) and they double as numbers: `True == 1`, so summing or averaging a mask counts matches instantly.

**Syntax:**
```python
arr > value            # boolean array of verdicts
(mask).sum()           # how many matched
(mask).mean()          # fraction that matched - instant rate!
```

In [ ]:
import numpy as np

scores = np.array([45, 82, 67, 91, 58])

print(">= 60    :", scores >= 60)
print("== 67    :", scores == 67)
print("in range :", (scores >= 60) & (scores <= 80))

pass_rate = (scores >= 60).mean()
print("pass rate:", pass_rate, "-", int(pass_rate * scores.size), "of", scores.size, "students")

## 4. Aggregations: One Number per Array

Aggregations collapse many values into one summary. Every method has an equivalent function form (`arr.sum()` == `np.sum(arr)`).

**Syntax:**
```python
arr.sum()      arr.mean()     arr.std()      arr.var()
arr.min()      arr.max()
np.median(arr)                np.percentile(arr, q)
```

In [ ]:
import numpy as np

salaries = np.array([35, 48, 52, 61, 90])   # thousands of taka

print("sum :", salaries.sum())
print("min :", salaries.min(), "| max :", salaries.max())
print("mean:", salaries.mean())
print("var :", salaries.var())
print("std :", round(salaries.std(), 2))

In [ ]:
import numpy as np

salaries = np.array([35, 48, 52, 61, 90, 95, 300])   # one CEO skews everything

print("mean  :", salaries.mean())        # dragged way up by the outlier
print("median:", np.median(salaries))    # the middle person - robust to outliers
print("quartiles & p90:", np.percentile(salaries, [25, 50, 90]))

> 🔍 **Under the Hood:** `np.sum` does not simply add left-to-right. It uses **pairwise summation**: the buffer is recursively split into blocks that are summed independently, so floating-point rounding errors stop accumulating linearly — it is both faster (cache-friendly) and MORE accurate than a naive Python loop over the same values. Aggregations also release Python's GIL internally while working on the raw buffer.

## 5. ⚠️ Axis Demystified: axis=0 vs axis=1

The single most confusing argument in NumPy — until you learn the trick: **the axis is the direction that gets COLLAPSED**, i.e. the dimension that disappears from the result's shape.

For a `(3 rows, 4 cols)` matrix:

```
axis=0 : collapse DOWN each column  -> one value per column  -> shape (4,)
axis=1 : collapse ACROSS each row   -> one value per row     -> shape (3,)
no axis: collapse EVERYTHING        -> one single number
```

So `m.sum(axis=0)` does NOT mean "sum the rows" — it means "sum DOWN the columns, removing the row axis".

In [ ]:
import numpy as np

m = np.arange(12).reshape(3, 4)     # 3 rows x 4 columns
print(m)
print()

col_totals = m.sum(axis=0)          # collapse DOWN: one total per column
print("sum(axis=0) ->", col_totals, "| shape:", col_totals.shape)
print("expected: [0+4+8, 1+5+9, 2+6+10, 3+7+11] =", 0 + 4 + 8, 1 + 5 + 9, 2 + 6 + 10, 3 + 7 + 11)

row_totals = m.sum(axis=1)          # collapse ACROSS: one total per row
print("sum(axis=1) ->", row_totals, "| shape:", row_totals.shape)
print("expected: [0+1+2+3, 4+5+6+7, 8+9+10+11] =", 6, 22, 38)

print("sum()       ->", m.sum())     # no axis: everything collapses to one number

In [ ]:
import numpy as np

grades = np.array([[78, 85, 90],     # 3 students x 3 subjects
                   [62, 71, 88],
                   [90, 93, 79]])

print("per-SUBJECT average (collapse students, axis=0):", grades.mean(axis=0))
print("per-STUDENT best   (collapse subjects, axis=1):", grades.max(axis=1))
print("keepdims keeps the axis alive for later broadcasting:",
      grades.mean(axis=1, keepdims=True).shape)

## 6. argmin / argmax: Where Is the Extreme?

`max` tells you the biggest VALUE; `argmax` tells you its POSITION (index). That index is usually what you actually want — which candidate won? Which feature matters most?

**Syntax:**
```python
arr.argmax()             # index of the maximum (whole array)
arr.argmin()             # index of the minimum
m.argmax(axis=1)         # per-row winners
```

In [ ]:
import numpy as np

votes = np.array([120, 340, 275, 90])
candidates = ["Amina", "Rafi", "Karim", "Nadia"]

winner_idx = votes.argmax()
print("winner index :", winner_idx, "->", candidates[winner_idx], "with", votes.max(), "votes")
print("loser index  :", votes.argmin(), "->", candidates[votes.argmin()])

quiz = np.array([[4, 9, 1],
                 [7, 2, 8]])
print("best subject per student (axis=1):", quiz.argmax(axis=1))

## 7. Running Totals: cumsum / cumprod

While `sum` collapses to one number, `cumsum` keeps a running tally — same shape as the input, each position holding the total so far. Ideal for cumulative sales, balances and growth curves.

**Syntax:**
```python
arr.cumsum()     # running total at each position
arr.cumprod()    # running product (compounding growth)
```

In [ ]:
import numpy as np

daily_sales = np.array([12, 9, 15, 7, 11])   # units sold Mon-Fri

print("daily          :", daily_sales)
print("running totals :", daily_sales.cumsum())

growth = np.array([1, 2, 3, 4])
print("running product:", growth.cumprod())   # compounding effect

## 8. any / all: Yes-or-No Questions

`.any()` asks "is at least ONE element True?" and `.all()` asks "are ALL elements True?" They turn whole arrays into a single decision — perfect for sanity checks before processing.

**Syntax:**
```python
(cond).any()    # True if any element satisfies cond
(cond).all()    # True only if every element satisfies cond
```

In [ ]:
import numpy as np

temps = np.array([31.5, 33.8, 30.2, 34.1])

print("any temp above 33?  ", (temps > 33).any())
print("all temps above 30? ", (temps > 30).all())
print("any temp below 30?  ", (temps < 30).any())

empty = np.array([])
print(".any() on empty array:", empty.any())   # False - nothing satisfied

## 9. NaN: Missing Data and the nan-aware Functions

NaN (*Not a Number*) marks missing data. It is contagious: ANY arithmetic touching NaN produces NaN, so one broken sensor reading silently poisons your whole mean. The fix: either drop NaNs yourself or use the nan-aware variants that skip them.

**Syntax:**
```python
np.isnan(arr)              # boolean map of missing positions
arr[~np.isnan(arr)]        # manual removal
np.nanmean(arr)            # mean ignoring NaNs (also nansum, nanmin, nanstd...)
```

In [ ]:
import numpy as np

readings = np.array([36.5, np.nan, 37.1, 36.9])

print("mean:", readings.mean())          # nan - one bad reading ruins it all
print("sum :", readings.sum())           # nan again
print("missing at:", np.isnan(readings))
print("clean mean:", readings[~np.isnan(readings)].mean())   # DIY fix

In [ ]:
import numpy as np

readings = np.array([36.5, np.nan, 37.1, 36.9])

print("np.nanmean:", np.nanmean(readings))   # ignores the gap automatically
print("np.nansum :", np.nansum(readings))
print("np.nanmax :", np.nanmax(readings))

# The whole family exists: nanmin, nanstd, nanargmax, ...

## ⚠️ Common Mistakes & Gotchas

| Mistake | Problem | Fix |
|---|---|---|
| Thinking `axis=0` operates on rows | It collapses rows away — results are per COLUMN | Remember: the axis is what disappears |
| Averaging data containing NaN | Result is silently `nan`, poisoning dashboards | `np.nanmean(...)` or clean with `~np.isnan(x)` first |
| Using `mean()` on heavily skewed data | Outliers drag it far from the typical value | Report `np.median` / percentiles alongside |
| Comparing float arrays with `==` | Rounding makes mathematically equal arrays differ | `np.allclose(a, b)` |
| Subtracting stats without `keepdims` | Shape `(n,)` stats broadcast along the WRONG axis | Use `keepdims=True` or `[:, np.newaxis]` deliberately |

## 💡 Best Practices & Pro Tips

- Prefer the method form `arr.mean()` over `np.mean(arr)` — identical behavior, reads cleaner.
- `(condition).mean()` is the fastest way to compute rates and proportions.
- For salaries, latencies, house prices — anything skewed — lead with median and percentiles, not mean.
- Run `.any()`/`.all()` assertions on inputs ("no negatives", "no NaN") at the top of pipelines; failures surface early and loudly.
- **AI-engineering relevance:** loss functions ARE aggregations — MSE is literally `((pred - target) ** 2).mean()`, accuracy is `(preds == labels).mean()`. And NaN hygiene is half of practical ML data cleaning.

## 📌 Summary

| Tool | What it does | Example |
|---|---|---|
| `+ - * / ** % //` | Element-wise arithmetic | `a * b`, `a + 100` |
| `np.sqrt / exp / log / abs / sin` | Universal functions, element-wise | `np.sqrt(x)` |
| `np.round / floor / ceil / trunc` | Rounding (round uses banker's rule) | `np.round(v)` |
| `arr > v` | Boolean array of comparisons | `(scores >= 60)` |
| `.sum() .mean() .std() .var()` | Aggregations to one number | `x.mean()` |
| `.min() .max()` | Extremes | `x.max()` |
| `np.median / percentile` | Robust / ranked summaries | `np.percentile(x, 75)` |
| `axis=0` / `axis=1` | Collapse down columns / across rows | `m.sum(axis=0)` |
| `.argmax() / .argmin()` | INDEX of the extreme | `votes.argmax()` |
| `.cumsum() / .cumprod()` | Running totals / products | `sales.cumsum()` |
| `.any() / .all()` | At least one / every element true | `(t > 33).any()` |
| `np.isnan` + `nanmean/nansum...` | Detect / skip missing values | `np.nanmean(r)` |

Key takeaways:
- Ufuncs apply element-wise in compiled C — never loop for math.
- The axis is the collapsed dimension: `axis=0` → one result per column, `axis=1` → per row.
- `argmax` returns a position, not a value.
- NaN spreads through every calculation; reach for the nan-family when data has gaps.

## 🔗 Next Lesson

- Continue to **[06_Sorting_Searching_Filtering](../06_Sorting_Searching_Filtering/notes.ipynb)** — sorting in place vs copies, argsort top-k, the three faces of `np.where`, and more.